# 01 — Manifest ASVspoof 2019 LA (train/dev) e controlli sull'audio

1. il manifest di train e dev, costruito **dai protocolli CM** 
2. i controlli di integrità (conteggi, speaker disgiunti, file mancanti o in più);
3. un test di decodifica dei FLAC con tre decoder (soundfile, ffmpeg, Parselmouth);
4. le statistiche delle durate e un controllo di *shortcut* 
5. una stima dello spazio disco necessario per le feature.

**Convenzione etichette:** `y_bonafide = 1` per bona fide, `0` per spoof.

In [1]:
import os, sys, re, json, math, shutil, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve

INPUT_ROOT = Path(os.environ.get("INPUT_ROOT", "/kaggle/input"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/manifests"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_DECODE_SAMPLES = 300      # file controllati con tutti e tre i decoder
N_WORKERS = 16              # thread per leggere gli header FLAC

# Valori attesi, verificati sui protocolli e sui conteggi delle cartelle
EXPECTED = {
    "train": {"rows": 25380, "bonafide": 2580, "spoof": 22800, "speakers": 20,
              "attacks": {"A01", "A02", "A03", "A04", "A05", "A06"}, "prefix": "LA_T_", "extra_files": 0},
    "dev":   {"rows": 24844, "bonafide": 2548, "spoof": 22296, "speakers": 20,
              "attacks": {"A01", "A02", "A03", "A04", "A05", "A06"}, "prefix": "LA_D_", "extra_files": 142},
}
PROTOCOL_PATTERNS = {"train": r"cm[._]train[._]trn", "dev": r"cm[._]dev[._]trl"}
AUDIO_SUBDIRS = {"train": "ASVspoof2019_LA_train/flac", "dev": "ASVspoof2019_LA_dev/flac"}

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)})
    print(f"[{'OK ' if ok else 'ERR'}] {name}" + (f"  ->  {detail}" if detail != "" else ""))

print("Python", sys.version.split()[0], "| soundfile", sf.__version__, "| libsndfile", sf.__libsndfile_version__)

Python 3.12.13 | soundfile 0.13.1 | libsndfile 1.2.2


In [16]:
def find_la_roots(root):
    hits = []
    for dirpath, dirnames, _ in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "flac"]
        if "ASVspoof2019_LA_cm_protocols" in dirnames:
            hits.append(Path(dirpath))
    return hits

roots = find_la_roots(INPUT_ROOT)
print("Candidati:", roots)
if len(roots) != 1:
    raise RuntimeError(f"Attesa esattamente una cartella LA, trovate {len(roots)}: {roots}")
LA_ROOT = roots[0]
PROTO_DIR = LA_ROOT / "ASVspoof2019_LA_cm_protocols"
print("LA_ROOT =", LA_ROOT)
print("Protocolli presenti:", sorted(p.name for p in PROTO_DIR.iterdir()))
for split, sub in AUDIO_SUBDIRS.items():
    check(f"cartella audio {split} esiste", (LA_ROOT / sub).is_dir(), LA_ROOT / sub)

Candidati: [PosixPath('/kaggle/input/datasets/angelopaldino/asvspoof2019-train-dev/LA')]
LA_ROOT = /kaggle/input/datasets/angelopaldino/asvspoof2019-train-dev/LA
Protocolli presenti: ['ASVspoof2019.LA.cm.dev.trl.txt', 'ASVspoof2019.LA.cm.eval.trl.txt', 'ASVspoof2019.LA.cm.train.trn.txt']
[OK ] cartella audio train esiste  ->  /kaggle/input/datasets/angelopaldino/asvspoof2019-train-dev/LA/ASVspoof2019_LA_train/flac
[OK ] cartella audio dev esiste  ->  /kaggle/input/datasets/angelopaldino/asvspoof2019-train-dev/LA/ASVspoof2019_LA_dev/flac


# LETTURA PROTOCOLLI CM

In [17]:
COLS = ["speaker", "utt_id", "unused", "attack", "label"]

def resolve_protocol(split):
    pat = re.compile(PROTOCOL_PATTERNS[split])
    found = [p for p in PROTO_DIR.iterdir() if pat.search(p.name)]
    if len(found) != 1:
        raise RuntimeError(f"{split}: attesi 1 file di protocollo, trovati {found}")
    return found[0]

def load_protocol(split):
    path = resolve_protocol(split)
    df = pd.read_csv(path, sep=r"\s+", header=None, dtype=str, keep_default_na=False, engine="python")
    if df.shape[1] != len(COLS):
        raise ValueError(f"{path.name}: {df.shape[1]} colonne invece di {len(COLS)}")
    df.columns = COLS
    df.insert(0, "split", split)
    print(f"{split}: {path.name}, {len(df)} righe")
    return df

frames = []
for split in ["train", "dev"]:
    df = load_protocol(split)
    exp = EXPECTED[split]
    vc = df["label"].value_counts().to_dict()
    check(f"{split}: numero righe", len(df) == exp["rows"], f"{len(df)} (attese {exp['rows']})")
    check(f"{split}: etichette ammesse", set(vc) <= {"bonafide", "spoof"}, vc)
    check(f"{split}: bona fide / spoof", (vc.get("bonafide", 0), vc.get("spoof", 0)) == (exp["bonafide"], exp["spoof"]), vc)
    check(f"{split}: colonna 3 sempre '-'", (df["unused"] == "-").all(), list(df["unused"].unique()[:5]))
    check(f"{split}: attacco '-' <=> bona fide", ((df["attack"] == "-") == (df["label"] == "bonafide")).all())
    atk = set(df.loc[df["label"] == "spoof", "attack"])
    check(f"{split}: insieme attacchi", atk == exp["attacks"], sorted(atk))
    check(f"{split}: prefisso nomi file", df["utt_id"].str.startswith(exp["prefix"]).all(), exp["prefix"])
    check(f"{split}: nessun duplicato", not df["utt_id"].duplicated().any())
    check(f"{split}: numero speaker", df["speaker"].nunique() == exp["speakers"], df["speaker"].nunique())
    frames.append(df)

manifest = pd.concat(frames, ignore_index=True).drop(columns="unused")
manifest["attack"] = manifest["attack"].replace("-", "bonafide")
manifest["y_bonafide"] = (manifest["label"] == "bonafide").astype(np.int8)
manifest["relpath"] = [f"{AUDIO_SUBDIRS[s]}/{u}.flac" for s, u in zip(manifest["split"], manifest["utt_id"])]

spk = {s: set(manifest.loc[manifest["split"] == s, "speaker"]) for s in ["train", "dev"]}
check("speaker disgiunti train/dev", not (spk["train"] & spk["dev"]), sorted(spk["train"] & spk["dev"]))

# Speaker del dev presenti solo come bona fide (non-target): noti dall'analisi dei protocolli
only_bona = (manifest[manifest["split"] == "dev"].groupby("speaker")["y_bonafide"].min() == 1)
print("Speaker dev con solo bona fide:", sorted(only_bona[only_bona].index))
manifest.head()

train: ASVspoof2019.LA.cm.train.trn.txt, 25380 righe
[OK ] train: numero righe  ->  25380 (attese 25380)
[OK ] train: etichette ammesse  ->  {'spoof': 22800, 'bonafide': 2580}
[OK ] train: bona fide / spoof  ->  {'spoof': 22800, 'bonafide': 2580}
[OK ] train: colonna 3 sempre '-'  ->  ['-']
[OK ] train: attacco '-' <=> bona fide
[OK ] train: insieme attacchi  ->  ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
[OK ] train: prefisso nomi file  ->  LA_T_
[OK ] train: nessun duplicato
[OK ] train: numero speaker  ->  20
dev: ASVspoof2019.LA.cm.dev.trl.txt, 24844 righe
[OK ] dev: numero righe  ->  24844 (attese 24844)
[OK ] dev: etichette ammesse  ->  {'spoof': 22296, 'bonafide': 2548}
[OK ] dev: bona fide / spoof  ->  {'spoof': 22296, 'bonafide': 2548}
[OK ] dev: colonna 3 sempre '-'  ->  ['-']
[OK ] dev: attacco '-' <=> bona fide
[OK ] dev: insieme attacchi  ->  ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
[OK ] dev: prefisso nomi file  ->  LA_D_
[OK ] dev: nessun duplicato
[OK ] dev: numero sp

,split,speaker,utt_id,attack,label,y_bonafide,relpath
0,train,LA_0079,LA_T_1138215,bonafide,bonafide,1,ASVspoof2019_LA_train/flac/LA_T_1138215.flac
1,train,LA_0079,LA_T_1271820,bonafide,bonafide,1,ASVspoof2019_LA_train/flac/LA_T_1271820.flac
2,train,LA_0079,LA_T_1272637,bonafide,bonafide,1,ASVspoof2019_LA_train/flac/LA_T_1272637.flac
3,train,LA_0079,LA_T_1276960,bonafide,bonafide,1,ASVspoof2019_LA_train/flac/LA_T_1276960.flac
4,train,LA_0079,LA_T_1341447,bonafide,bonafide,1,ASVspoof2019_LA_train/flac/LA_T_1341447.flac


In [18]:
for split in ["train", "dev"]:
    on_disk = {f[:-5] for f in os.listdir(LA_ROOT / AUDIO_SUBDIRS[split]) if f.endswith(".flac")}
    in_proto = set(manifest.loc[manifest["split"] == split, "utt_id"])
    missing, extra = in_proto - on_disk, on_disk - in_proto
    check(f"{split}: file del protocollo tutti presenti", not missing, f"mancanti: {len(missing)} {sorted(missing)[:5]}")
    check(f"{split}: file in più = attesi", len(extra) == EXPECTED[split]["extra_files"], f"{len(extra)} (attesi {EXPECTED[split]['extra_files']})")
    if extra:
        non_enroll = [e for e in extra if not re.match(r"LA_[TD]_A\d+$", e)]
        check(f"{split}: i file in più sono tutti di enrollment (_A)", not non_enroll, non_enroll[:5])

[OK ] train: file del protocollo tutti presenti  ->  mancanti: 0 []
[OK ] train: file in più = attesi  ->  0 (attesi 0)
[OK ] dev: file del protocollo tutti presenti  ->  mancanti: 0 []
[OK ] dev: file in più = attesi  ->  142 (attesi 142)
[OK ] dev: i file in più sono tutti di enrollment (_A)  ->  []


# TEST SU UN CAMPIONE

In [19]:
try:
    import parselmouth
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth"], check=False)
    try:
        import parselmouth
    except ImportError:
        parselmouth = None
FFMPEG = shutil.which("ffmpeg")
print("parselmouth:", getattr(parselmouth, "__version__", "NON disponibile"), "| ffmpeg:", FFMPEG or "NON disponibile")

def read_ffmpeg(path):
    cmd = [FFMPEG, "-nostdin", "-v", "error", "-i", str(path), "-f", "s16le", "-acodec", "pcm_s16le", "-"]
    out = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(out, dtype=np.int16).astype(np.float32) / 32768.0

def decode_report(row):
    path = LA_ROOT / row.relpath
    rec = {"utt_id": row.utt_id, "split": row.split, "attack": row.attack}
    try:
        x_sf, sr = sf.read(path, dtype="float32", always_2d=False)
        rec.update(sf_ok=True, sr=sr, sf_ndim=x_sf.ndim, n_sf=len(x_sf))
    except Exception as e:
        rec.update(sf_ok=False, sf_err=repr(e)); x_sf = None
    if FFMPEG:
        try:
            x_ff = read_ffmpeg(path); rec.update(ff_ok=True, n_ff=len(x_ff))
            if x_sf is not None and len(x_ff) == len(x_sf):
                rec["maxdiff_sf_ff"] = float(np.max(np.abs(x_sf - x_ff)))
        except Exception as e:
            rec.update(ff_ok=False, ff_err=repr(e))
    if parselmouth is not None:
        try:
            snd = parselmouth.Sound(str(path)); x_pm = snd.values[0]
            rec.update(pm_ok=True, n_pm=len(x_pm), sr_pm=snd.sampling_frequency)
            if x_sf is not None and len(x_pm) == len(x_sf):
                rec["maxdiff_sf_pm"] = float(np.max(np.abs(x_sf - x_pm)))
        except Exception as e:
            rec.update(pm_ok=False, pm_err=repr(e))
    return rec

groups = manifest.groupby(["split", "attack"])
per_group = math.ceil(N_DECODE_SAMPLES / groups.ngroups)
sample = (manifest.sample(frac=1.0, random_state=SEED)
          .groupby(["split", "attack"], sort=False).head(per_group)
          .reset_index(drop=True))
print(f"Campione: {len(sample)} file ({per_group} per gruppo split x attacco)")
dec = pd.DataFrame([decode_report(r) for r in tqdm(sample.itertuples(), total=len(sample), desc="decode")])

check("soundfile legge tutti i file del campione", dec["sf_ok"].all(), (~dec["sf_ok"]).sum())
if "ff_ok" in dec:
    check("ffmpeg legge tutti i file del campione", dec["ff_ok"].all(), (~dec["ff_ok"]).sum())
    same_len = (dec["n_ff"] == dec["n_sf"]).mean()
    check("ffmpeg: stessa lunghezza di soundfile", same_len == 1.0, f"{same_len:.3f}")
    check("ffmpeg: stesso segnale di soundfile", (dec["maxdiff_sf_ff"].fillna(np.inf) == 0).all(), dec["maxdiff_sf_ff"].max())
if "pm_ok" in dec:
    check("Parselmouth legge tutti i file del campione", dec["pm_ok"].all(), (~dec["pm_ok"]).sum())
    same_len = (dec["n_pm"] == dec["n_sf"]).mean()
    check("Parselmouth: stessa lunghezza di soundfile", same_len == 1.0, f"{same_len:.3f}")
    check("Parselmouth: stesso segnale (tolleranza 1e-6)", (dec["maxdiff_sf_pm"].fillna(np.inf) < 1e-6).all(), dec["maxdiff_sf_pm"].max())
check("campione a 16 kHz mono", (dec["sr"] == 16000).all() and (dec["sf_ndim"] == 1).all())
err_cols = [c for c in dec.columns if c.endswith("_err")]
if err_cols:
    display(dec.loc[dec[err_cols].notna().any(axis=1), ["utt_id"] + err_cols].head(10))
dec.describe(include="all").T.head(20)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.9 MB/s eta 0:00:00
parselmouth: 0.4.7 | ffmpeg: /usr/bin/ffmpeg
Campione: 308 file (22 per gruppo split x attacco)


decode:   0%|          | 0/308 [00:00<?, ?it/s]

[OK ] soundfile legge tutti i file del campione  ->  0
[OK ] ffmpeg legge tutti i file del campione  ->  0
[OK ] ffmpeg: stessa lunghezza di soundfile  ->  1.000
[OK ] ffmpeg: stesso segnale di soundfile  ->  0.0
[OK ] Parselmouth legge tutti i file del campione  ->  0
[OK ] Parselmouth: stessa lunghezza di soundfile  ->  1.000
[OK ] Parselmouth: stesso segnale (tolleranza 1e-6)  ->  0.0
[OK ] campione a 16 kHz mono


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
utt_id,308,308,LA_T_9206457,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
split,308,2,train,154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
attack,308,7,A06,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sf_ok,308,1,True,308,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sr,308.0,NaN,NaN,NaN,16000.0,0.0,16000.0,16000.0,16000.0,16000.0,16000.0
sf_ndim,308.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
n_sf,308.0,NaN,NaN,NaN,56806.068182,23925.562147,15924.0,38932.25,55098.0,68674.25,173473.0
ff_ok,308,1,True,308,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_ff,308.0,NaN,NaN,NaN,56806.068182,23925.562147,15924.0,38932.25,55098.0,68674.25,173473.0
maxdiff_sf_ff,308.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# DURATE FILE

In [20]:
def header_info(relpath):
    try:
        i = sf.info(LA_ROOT / relpath)
        return (i.samplerate, i.channels, i.frames, i.subtype, "")
    except Exception as e:
        return (np.nan, np.nan, np.nan, "", repr(e))

with ThreadPoolExecutor(N_WORKERS) as ex:
    infos = list(tqdm(ex.map(header_info, manifest["relpath"]), total=len(manifest), desc="header"))

manifest[["sample_rate", "channels", "n_samples", "subtype", "read_error"]] = pd.DataFrame(infos, index=manifest.index)
manifest["duration_s"] = manifest["n_samples"] / manifest["sample_rate"]

check("header leggibili per tutti i file", (manifest["read_error"] == "").all(), (manifest["read_error"] != "").sum())
check("tutti a 16 kHz", (manifest["sample_rate"] == 16000).all(), manifest["sample_rate"].value_counts().to_dict())
check("tutti mono", (manifest["channels"] == 1).all(), manifest["channels"].value_counts().to_dict())
print("Subtype:", manifest["subtype"].value_counts().to_dict())
check("nessun file oltre 30 s (limite encoder Whisper)", (manifest["duration_s"] <= 30).all(), f"max {manifest['duration_s'].max():.2f} s")

pct = [.01, .05, .25, .5, .75, .95, .99]
display(manifest.groupby(["split", "label"])["duration_s"].describe(percentiles=pct).round(2))
display(manifest.groupby(["split", "attack"])["duration_s"].median().unstack(0).round(2))
for thr in [2, 4, 6, 8]:
    frac = manifest.groupby(["split", "label"])["duration_s"].apply(lambda d: (d < thr).mean())
    print(f"quota < {thr} s:", frac.round(3).to_dict())

header:   0%|          | 0/50224 [00:00<?, ?it/s]

[OK ] header leggibili per tutti i file  ->  0
[OK ] tutti a 16 kHz  ->  {16000: 50224}
[OK ] tutti mono  ->  {1: 50224}
Subtype: {'PCM_16': 50224}
[OK ] nessun file oltre 30 s (limite encoder Whisper)  ->  max 13.19 s


count  mean   std   min    1%    5%   25%   50%   75%   95%  \
split label                                                                     
dev   bonafide   2548.0  3.51  1.09  1.28  1.66  2.00  2.73  3.39  4.14  5.42   
      spoof     22296.0  3.47  1.49  0.70  1.09  1.50  2.41  3.27  4.25  6.25   
train bonafide   2580.0  3.39  0.99  1.36  1.70  2.05  2.69  3.24  3.92  5.16   
      spoof     22800.0  3.43  1.46  0.65  1.16  1.56  2.39  3.20  4.14  6.17   

                 99%    max  
split label                  
dev   bonafide  6.65  11.39  
      spoof     8.33  11.59  
train bonafide  6.48  11.13  
      spoof     8.20  13.19

split,dev,train
attack,,
A01,2.44,2.38
A02,3.89,3.82
A03,2.90,2.91
A04,2.74,2.72
A05,3.54,3.49
A06,3.54,3.49
bonafide,3.39,3.24


quota < 2 s: {('dev', 'bonafide'): 0.05, ('dev', 'spoof'): 0.147, ('train', 'bonafide'): 0.042, ('train', 'spoof'): 0.141}
quota < 4 s: {('dev', 'bonafide'): 0.699, ('dev', 'spoof'): 0.701, ('train', 'bonafide'): 0.765, ('train', 'spoof'): 0.72}
quota < 6 s: {('dev', 'bonafide'): 0.978, ('dev', 'spoof'): 0.939, ('train', 'bonafide'): 0.981, ('train', 'spoof'): 0.943}
quota < 8 s: {('dev', 'bonafide'): 0.997, ('dev', 'spoof'): 0.986, ('train', 'bonafide'): 0.998, ('train', 'spoof'): 0.988}


In [21]:
BYTES_PER_FRAME = (512 + 768 + 5) * 2
rows = []
for lmax in [4, 6, 8, None]:
    dur = manifest["duration_s"] if lmax is None else manifest["duration_s"].clip(upper=lmax)
    gb = (np.ceil(dur * 50) * BYTES_PER_FRAME).groupby(manifest["split"]).sum() / 1e9
    rows.append({"Lmax_s": lmax or "intera", **{f"{s}_GB": round(gb[s], 2) for s in gb.index}, "tot_GB": round(gb.sum(), 2)})
pd.DataFrame(rows)

,Lmax_s,dev_GB,train_GB,tot_GB
0,4,9.95,10.10,20.05
1,6,10.91,10.99,21.91
2,8,11.10,11.17,22.26
3,intera,11.14,11.21,22.34
